In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, accuracy_score

from sklearn.inspection import permutation_importance


In [ ]:
df = pd.read_csv("loan_data_set.csv")


In [ ]:
print(df.to_string)


In [ ]:
<bound method DataFrame.to_string of       Loan_ID  Gender Married Dependents     Education Self_Employed  \
0    LP001002    Male      No          0      Graduate            No   
1    LP001003    Male     Yes          1      Graduate            No   
2    LP001005    Male     Yes          0      Graduate           Yes   
3    LP001006    Male     Yes          0  Not Graduate            No   
4    LP001008    Male      No          0      Graduate            No   
..        ...     ...     ...        ...           ...           ...   
609  LP002978  Female      No          0      Graduate            No   
610  LP002979    Male     Yes         3+      Graduate            No   
611  LP002983    Male     Yes          1      Graduate            No   
612  LP002984    Male     Yes          2      Graduate            No   
613  LP002990  Female      No          0      Graduate           Yes   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0               5849                0.0         NaN             360.0   
1               4583             1508.0       128.0             360.0   
2               3000                0.0        66.0             360.0   
3               2583             2358.0       120.0             360.0   
4               6000                0.0       141.0             360.0   
..               ...                ...         ...               ...   
609             2900                0.0        71.0             360.0   
610             4106                0.0        40.0             180.0   
611             8072              240.0       253.0             360.0   
612             7583                0.0       187.0             360.0   
613             4583                0.0       133.0             360.0   

     Credit_History Property_Area Loan_Status  
0               1.0         Urban           Y  
1               1.0         Rural           N  
2               1.0         Urban           Y  
3               1.0         Urban           Y  
4               1.0         Urban           Y  
..              ...           ...         ...  
609             1.0         Rural           Y  
610             1.0         Rural           Y  
611             1.0         Urban           Y  
612             1.0         Urban           Y  
613             0.0     Semiurban           N  

[614 rows x 13 columns]>


In [ ]:
df


In [ ]:
print(df.head())


In [ ]:
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N  
2             1.0         Urban           Y  
3             1.0         Urban           Y  
4             1.0         Urban           Y  


In [ ]:
print(df.info())


In [ ]:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB
None


In [ ]:
print(df.describe())


In [ ]:
       ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
count       614.000000         614.000000  592.000000         600.00000   
mean       5403.459283        1621.245798  146.412162         342.00000   
std        6109.041673        2926.248369   85.587325          65.12041   
min         150.000000           0.000000    9.000000          12.00000   
25%        2877.500000           0.000000  100.000000         360.00000   
50%        3812.500000        1188.500000  128.000000         360.00000   
75%        5795.000000        2297.250000  168.000000         360.00000   
max       81000.000000       41667.000000  700.000000         480.00000   

       Credit_History  
count      564.000000  
mean         0.842199  
std          0.364878  
min          0.000000  
25%          1.000000  
50%          1.000000  
75%          1.000000  
max          1.000000  


In [ ]:
print("\n--- Missing values (count and percent) ---")
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)
display(missing)


In [ ]:

--- Missing values (count and percent) ---


In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print("\nCategorical columns detected:", cat_cols)
for col in cat_cols:
    unique_vals = df[col].dropna().unique()
    print(f"\nColumn: {col} - unique values ({len(unique_vals)}):\n", unique_vals)


In [ ]:

Categorical columns detected: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

Column: Loan_ID - unique values (614):
 ['LP001002' 'LP001003' 'LP001005' 'LP001006' 'LP001008' 'LP001011'
 'LP001013' 'LP001014' 'LP001018' 'LP001020' 'LP001024' 'LP001027'
 'LP001028' 'LP001029' 'LP001030' 'LP001032' 'LP001034' 'LP001036'
 'LP001038' 'LP001041' 'LP001043' 'LP001046' 'LP001047' 'LP001050'
 'LP001052' 'LP001066' 'LP001068' 'LP001073' 'LP001086' 'LP001087'
 'LP001091' 'LP001095' 'LP001097' 'LP001098' 'LP001100' 'LP001106'
 'LP001109' 'LP001112' 'LP001114' 'LP001116' 'LP001119' 'LP001120'
 'LP001123' 'LP001131' 'LP001136' 'LP001137' 'LP001138' 'LP001144'
 'LP001146' 'LP001151' 'LP001155' 'LP001157' 'LP001164' 'LP001179'
 'LP001186' 'LP001194' 'LP001195' 'LP001197' 'LP001198' 'LP001199'
 'LP001205' 'LP001206' 'LP001207' 'LP001213' 'LP001222' 'LP001225'
 'LP001228' 'LP001233' 'LP001238' 'LP001241' 'LP001243' 'LP001245'
 'LP001248' 'LP001250' 'LP001253' 'LP001255' 'LP001256' 'LP001259'
 'LP001263' 'LP001264' 'LP001265' 'LP001266' 'LP001267' 'LP001273'
 'LP001275' 'LP001279' 'LP001280' 'LP001282' 'LP001289' 'LP001310'
 'LP001316' 'LP001318' 'LP001319' 'LP001322' 'LP001325' 'LP001326'
 'LP001327' 'LP001333' 'LP001334' 'LP001343' 'LP001345' 'LP001349'
 'LP001350' 'LP001356' 'LP001357' 'LP001367' 'LP001369' 'LP001370'
 'LP001379' 'LP001384' 'LP001385' 'LP001387' 'LP001391' 'LP001392'
 'LP001398' 'LP001401' 'LP001404' 'LP001405' 'LP001421' 'LP001422'
 'LP001426' 'LP001430' 'LP001431' 'LP001432' 'LP001439' 'LP001443'
 'LP001448' 'LP001449' 'LP001451' 'LP001465' 'LP001469' 'LP001473'
 'LP001478' 'LP001482' 'LP001487' 'LP001488' 'LP001489' 'LP001491'
 'LP001492' 'LP001493' 'LP001497' 'LP001498' 'LP001504' 'LP001507'
 'LP001508' 'LP001514' 'LP001516' 'LP001518' 'LP001519' 'LP001520'
 'LP001528' 'LP001529' 'LP001531' 'LP001532' 'LP001535' 'LP001536'
 'LP001541' 'LP001543' 'LP001546' 'LP001552' 'LP001560' 'LP001562'
 'LP001565' 'LP001570' 'LP001572' 'LP001574' 'LP001577' 'LP001578'
 'LP001579' 'LP001580' 'LP001581' 'LP001585' 'LP001586' 'LP001594'
 'LP001603' 'LP001606' 'LP001608' 'LP001610' 'LP001616' 'LP001630'
 'LP001633' 'LP001634' 'LP001636' 'LP001637' 'LP001639' 'LP001640'
 'LP001641' 'LP001643' 'LP001644' 'LP001647' 'LP001653' 'LP001656'
 'LP001657' 'LP001658' 'LP001664' 'LP001665' 'LP001666' 'LP001669'
 'LP001671' 'LP001673' 'LP001674' 'LP001677' 'LP001682' 'LP001688'
 'LP001691' 'LP001692' 'LP001693' 'LP001698' 'LP001699' 'LP001702'
 'LP001708' 'LP001711' 'LP001713' 'LP001715' 'LP001716' 'LP001720'
 'LP001722' 'LP001726' 'LP001732' 'LP001734' 'LP001736' 'LP001743'
 'LP001744' 'LP001749' 'LP001750' 'LP001751' 'LP001754' 'LP001758'
 'LP001760' 'LP001761' 'LP001765' 'LP001768' 'LP001770' 'LP001776'
 'LP001778' 'LP001784' 'LP001786' 'LP001788' 'LP001790' 'LP001792'
 'LP001798' 'LP001800' 'LP001806' 'LP001807' 'LP001811' 'LP001813'
 'LP001814' 'LP001819' 'LP001824' 'LP001825' 'LP001835' 'LP001836'
 'LP001841' 'LP001843' 'LP001844' 'LP001846' 'LP001849' 'LP001854'
 'LP001859' 'LP001864' 'LP001865' 'LP001868' 'LP001870' 'LP001871'
 'LP001872' 'LP001875' 'LP001877' 'LP001882' 'LP001883' 'LP001884'
 'LP001888' 'LP001891' 'LP001892' 'LP001894' 'LP001896' 'LP001900'
 'LP001903' 'LP001904' 'LP001907' 'LP001908' 'LP001910' 'LP001914'
 'LP001915' 'LP001917' 'LP001922' 'LP001924' 'LP001925' 'LP001926'
 'LP001931' 'LP001935' 'LP001936' 'LP001938' 'LP001940' 'LP001945'
 'LP001947' 'LP001949' 'LP001953' 'LP001954' 'LP001955' 'LP001963'
 'LP001964' 'LP001972' 'LP001974' 'LP001977' 'LP001978' 'LP001990'
 'LP001993' 'LP001994' 'LP001996' 'LP001998' 'LP002002' 'LP002004'
 'LP002006' 'LP002008' 'LP002024' 'LP002031' 'LP002035' 'LP002036'
 'LP002043' 'LP002050' 'LP002051' 'LP002053' 'LP002054' 'LP002055'
 'LP002065' 'LP002067' 'LP002068' 'LP002082' 'LP002086' 'LP002087'
 'LP002097' 'LP002098' 'LP002100' 'LP002101' 'LP002103' 'LP002106'
 'LP002110' 'LP002112' 'LP002113' 'LP002114' 'LP002115' 'LP002116'
 'LP002119' 'LP002126' 'LP002128' 'LP002129' 'LP002130' 'LP002131'
 'LP002137' 'LP002138' 'LP002139' 'LP002140' 'LP002141' 'LP002142'
 'LP002143' 'LP002144' 'LP002149' 'LP002151' 'LP002158' 'LP002160'
 'LP002161' 'LP002170' 'LP002175' 'LP002178' 'LP002180' 'LP002181'
 'LP002187' 'LP002188' 'LP002190' 'LP002191' 'LP002194' 'LP002197'
 'LP002201' 'LP002205' 'LP002209' 'LP002211' 'LP002219' 'LP002223'
 'LP002224' 'LP002225' 'LP002226' 'LP002229' 'LP002231' 'LP002234'
 'LP002236' 'LP002237' 'LP002239' 'LP002243' 'LP002244' 'LP002250'
 'LP002255' 'LP002262' 'LP002263' 'LP002265' 'LP002266' 'LP002272'
 'LP002277' 'LP002281' 'LP002284' 'LP002287' 'LP002288' 'LP002296'
 'LP002297' 'LP002300' 'LP002301' 'LP002305' 'LP002308' 'LP002314'
 'LP002315' 'LP002317' 'LP002318' 'LP002319' 'LP002328' 'LP002332'
 'LP002335' 'LP002337' 'LP002341' 'LP002342' 'LP002345' 'LP002347'
 'LP002348' 'LP002357' 'LP002361' 'LP002362' 'LP002364' 'LP002366'
 'LP002367' 'LP002368' 'LP002369' 'LP002370' 'LP002377' 'LP002379'
 'LP002386' 'LP002387' 'LP002390' 'LP002393' 'LP002398' 'LP002401'
 'LP002403' 'LP002407' 'LP002408' 'LP002409' 'LP002418' 'LP002422'
 'LP002424' 'LP002429' 'LP002434' 'LP002435' 'LP002443' 'LP002444'
 'LP002446' 'LP002447' 'LP002448' 'LP002449' 'LP002453' 'LP002455'
 'LP002459' 'LP002467' 'LP002472' 'LP002473' 'LP002478' 'LP002484'
 'LP002487' 'LP002489' 'LP002493' 'LP002494' 'LP002500' 'LP002501'
 'LP002502' 'LP002505' 'LP002515' 'LP002517' 'LP002519' 'LP002522'
 'LP002524' 'LP002527' 'LP002529' 'LP002530' 'LP002531' 'LP002533'
 'LP002534' 'LP002536' 'LP002537' 'LP002541' 'LP002543' 'LP002544'
 'LP002545' 'LP002547' 'LP002555' 'LP002556' 'LP002560' 'LP002562'
 'LP002571' 'LP002582' 'LP002585' 'LP002586' 'LP002587' 'LP002588'
 'LP002600' 'LP002602' 'LP002603' 'LP002606' 'LP002615' 'LP002618'
 'LP002619' 'LP002622' 'LP002624' 'LP002625' 'LP002626' 'LP002634'
 'LP002637' 'LP002640' 'LP002643' 'LP002648' 'LP002652' 'LP002659'
 'LP002670' 'LP002682' 'LP002683' 'LP002684' 'LP002689' 'LP002690'
 'LP002692' 'LP002693' 'LP002697' 'LP002699' 'LP002705' 'LP002706'
 'LP002714' 'LP002716' 'LP002717' 'LP002720' 'LP002723' 'LP002729'
 'LP002731' 'LP002732' 'LP002734' 'LP002738' 'LP002739' 'LP002740'
 'LP002741' 'LP002743' 'LP002753' 'LP002755' 'LP002757' 'LP002767'
 'LP002768' 'LP002772' 'LP002776' 'LP002777' 'LP002778' 'LP002784'
 'LP002785' 'LP002788' 'LP002789' 'LP002792' 'LP002794' 'LP002795'
 'LP002798' 'LP002804' 'LP002807' 'LP002813' 'LP002820' 'LP002821'
 'LP002832' 'LP002833' 'LP002836' 'LP002837' 'LP002840' 'LP002841'
 'LP002842' 'LP002847' 'LP002855' 'LP002862' 'LP002863' 'LP002868'
 'LP002872' 'LP002874' 'LP002877' 'LP002888' 'LP002892' 'LP002893'
 'LP002894' 'LP002898' 'LP002911' 'LP002912' 'LP002916' 'LP002917'
 'LP002925' 'LP002926' 'LP002928' 'LP002931' 'LP002933' 'LP002936'
 'LP002938' 'LP002940' 'LP002941' 'LP002943' 'LP002945' 'LP002948'
 'LP002949' 'LP002950' 'LP002953' 'LP002958' 'LP002959' 'LP002960'
 'LP002961' 'LP002964' 'LP002974' 'LP002978' 'LP002979' 'LP002983'
 'LP002984' 'LP002990']

Column: Gender - unique values (2):
 ['Male' 'Female']

Column: Married - unique values (2):
 ['No' 'Yes']

Column: Dependents - unique values (4):
 ['0' '1' '2' '3+']

Column: Education - unique values (2):
 ['Graduate' 'Not Graduate']

Column: Self_Employed - unique values (2):
 ['No' 'Yes']

Column: Property_Area - unique values (3):
 ['Urban' 'Rural' 'Semiurban']

Column: Loan_Status - unique values (2):
 ['Y' 'N']


In [ ]:
print("Before removing duplicates:", df.shape)
df.drop_duplicates(inplace=True)
print("After removing duplicates:", df.shape)


In [ ]:
Before removing duplicates: (614, 13)
After removing duplicates: (614, 13)


In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
print("\nNumeric columns for outlier checks:", num_cols)


In [ ]:

Numeric columns for outlier checks: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']


In [ ]:
import os
PLOT_DIR = 'plots'
os.makedirs(PLOT_DIR, exist_ok=True)

plt.figure(figsize=(12, 4 * len(num_cols)))
for i, col in enumerate(num_cols):
    plt.subplot(len(num_cols), 1, i + 1)
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot - {col}")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/boxplots_numeric.png")
plt.show()


In [ ]:
plt.figure(figsize=(12, 4 * len(num_cols)))
for i, col in enumerate(num_cols):
    plt.subplot(len(num_cols), 1, i + 1)
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f"Distribution - {col}")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/distributions_numeric.png")
plt.show()


In [ ]:
print(df.isnull().sum())


In [ ]:
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
cat_cols = df.select_dtypes(exclude=['number']).columns.tolist()

# Numeric → median
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical → "missing"
df[cat_cols] = df[cat_cols].fillna("missing")


In [ ]:
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
# remove target from features list
if 'Loan_Status' in num_cols:
    num_cols.remove('Loan_Status')

cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)


In [ ]:
Numeric columns: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
Categorical columns: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']


In [ ]:
if df["Loan_Status"].dtype == 'object':
    df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})

y = df["Loan_Status"]
X = df.drop(columns=["Loan_Status"])


In [ ]:
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()


In [ ]:
num_pipe_scaled = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

cat_pipe_ohe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess_ohe = ColumnTransformer([
    ('num', num_pipe_scaled, num_cols),
    ('cat', cat_pipe_ohe, cat_cols)
])


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

cat_pipe_label = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('label', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocess_label = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', cat_pipe_label, cat_cols)
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


In [ ]:
X_train_processed_label = preprocess_label.fit_transform(X_train)
X_test_processed_label = preprocess_label.transform(X_test)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

models = {
    "Logistic Regression": Pipeline(
        [("pre", preprocess_ohe), ("clf", LogisticRegression(max_iter=2000, solver="liblinear"))]
    ),
    "Decision Tree": Pipeline([("pre", preprocess_label), ("clf", DecisionTreeClassifier(max_depth=5, random_state=42))]),
    "Random Forest": Pipeline(
        [("pre", preprocess_label), ("clf", RandomForestClassifier(n_estimators=200, random_state=42))]
    ),
    "Gradient Boosting": Pipeline(
        [("pre", preprocess_ohe), ("clf", GradientBoostingClassifier(n_estimators=200, learning_rate=0.05))]
    ),
}


In [ ]:
print("\n--------------------")
print("MODEL ACCURACY RESULTS")
print("--------------------\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("ROC-AUC:", roc_auc_score(y_test, proba))
    print("\nClassification Report:\n", classification_report(y_test, pred))


In [ ]:

--------------------
MODEL ACCURACY RESULTS
--------------------


=== Logistic Regression ===
Accuracy: 0.8486486486486486
ROC-AUC: 0.8186261200108608

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.57      0.70        58
           1       0.83      0.98      0.90       127

    accuracy                           0.85       185
   macro avg       0.87      0.77      0.80       185
weighted avg       0.86      0.85      0.84       185


=== Decision Tree ===
Accuracy: 0.772972972972973
ROC-AUC: 0.7327586206896552

Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.57      0.61        58
           1       0.81      0.87      0.84       127

    accuracy                           0.77       185
   macro avg       0.74      0.72      0.73       185
weighted avg       0.77      0.77      0.77       185


=== Random Forest ===
Accuracy: 0.7945945945945946
ROC-AUC: 0.8106842248167254

Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.62      0.65        58
           1       0.83      0.87      0.85       127

    accuracy                           0.79       185
   macro avg       0.76      0.75      0.75       185
weighted avg       0.79      0.79      0.79       185


=== Gradient Boosting ===
Accuracy: 0.8432432432432433
ROC-AUC: 0.8008417051316862

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.55      0.69        58
           1       0.83      0.98      0.90       127

    accuracy                           0.84       185
   macro avg       0.87      0.76      0.79       185
weighted avg       0.85      0.84      0.83       185



In [ ]:
sns.countplot(x='Loan_Status', data=df)
plt.title("Loan Status Distribution")
plt.show()


In [ ]:
sns.scatterplot(x=df['ApplicantIncome'], y=df['LoanAmount'], hue=df['Loan_Status'])
plt.title("Applicant Income vs Loan Amount")
plt.show()


In [ ]:
sns.countplot(x='Gender', hue='Loan_Status', data=df)
plt.title("Loan Approval by Gender")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
df['Loan_Status'] = df['Loan_Status'].map({'Y':1, 'N':0})
y = df['Loan_Status']
X = df.drop('Loan_Status', axis=1)


In [ ]:
num_features = X.select_dtypes(include=['number']).columns
cat_features = X.select_dtypes(exclude=['number']).columns

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
])


In [ ]:
y.isna().sum()


In [ ]:
np.int64(614)

In [ ]:
log_pipe = Pipeline(steps=[
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, solver='liblinear'))
])

param_log = {'clf__C': [0.1, 1, 10]}

log_cv = GridSearchCV(log_pipe, param_log, cv=3, scoring='f1')
log_cv.fit(X_train, y_train)

log_pred = log_cv.predict(X_test)


In [ ]:
rf_pipe = Pipeline(steps=[
    ('pre', preprocessor),
    ('clf', RandomForestClassifier())
])

param_rf = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [3, 5, 10]
}

rf_cv = GridSearchCV(rf_pipe, param_rf, cv=3, scoring='f1')
rf_cv.fit(X_train, y_train)

rf_pred = rf_cv.predict(X_test)


In [ ]:
gb_pipe = Pipeline(steps=[
    ('pre', preprocessor),
    ('clf', GradientBoostingClassifier())
])

param_gb = {
    'clf__n_estimators': [100, 200],
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_depth': [2, 3]
}

gb_cv = GridSearchCV(gb_pipe, param_gb, cv=3, scoring='f1')
gb_cv.fit(X_train, y_train)

gb_pred = gb_cv.predict(X_test)


In [ ]:
def evaluate(name, model, predictions):
    print("\n====", name, "====")
    print(classification_report(y_test, predictions))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    print("AUC:", auc)

evaluate("Logistic Regression", log_cv, log_pred)
evaluate("Random Forest", rf_cv, rf_pred)
evaluate("Gradient Boosting", gb_cv, gb_pred)


In [ ]:

==== Logistic Regression ====
              precision    recall  f1-score   support

           0       0.92      0.57      0.70        58
           1       0.83      0.98      0.90       127

    accuracy                           0.85       185
   macro avg       0.87      0.77      0.80       185
weighted avg       0.86      0.85      0.84       185

AUC: 0.8186261200108608

==== Random Forest ====
              precision    recall  f1-score   support

           0       0.93      0.24      0.38        58
           1       0.74      0.99      0.85       127

    accuracy                           0.76       185
   macro avg       0.84      0.62      0.62       185
weighted avg       0.80      0.76      0.70       185

AUC: 0.806679337496606

==== Gradient Boosting ====
              precision    recall  f1-score   support

           0       0.94      0.55      0.70        58
           1       0.83      0.98      0.90       127

    accuracy                           0.85       185
   macro avg       0.88      0.77      0.80       185
weighted avg       0.86      0.85      0.84       185

AUC: 0.8095302742329623


In [ ]:
cm = confusion_matrix(y_test, log_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
rf_final = rf_cv.best_estimator_

result = permutation_importance(rf_final, X_test, y_test, n_repeats=10, random_state=42)

sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(10, 6))
plt.barh(X.columns[sorted_idx], result.importances_mean[sorted_idx])
plt.title("Permutation Feature Importance (Random Forest)")
plt.show()


In [ ]:
models = {
    "Logistic Regression": (log_pred, log_cv),
    "Random Forest": (rf_pred, rf_cv),
    "Gradient Boosting": (gb_pred, gb_cv)
}

summary = []

for name, (pred, model) in models.items():
    summary.append([
        name,
        accuracy_score(y_test, pred),
        roc_auc_score(y_test, model.predict_proba(X_test)[:,1]),
    ])

summary_df = pd.DataFrame(summary, columns=["Model", "Accuracy", "AUC"])
print(summary_df)


In [ ]:
                 Model  Accuracy       AUC
0  Logistic Regression  0.848649  0.818626
1        Random Forest  0.756757  0.806679
2    Gradient Boosting  0.848649  0.809530


In [ ]:
# chi-square for a categorical variable
from scipy.stats import chi2_contingency
ct = pd.crosstab(df['Gender'].fillna('missing'), df['Loan_Status'].fillna('N'))
chi2, p, dof, exp = chi2_contingency(ct)
print("Gender vs Loan_Status: chi2=%.3f p=%.5f" % (chi2, p))

# numeric correlation matrix
num_cols = ['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term','Credit_History']
print(df[num_cols].corr(method='pearson'))


In [ ]:
Gender vs Loan_Status: chi2=0.000 p=1.00000
                   ApplicantIncome  CoapplicantIncome  LoanAmount  \
ApplicantIncome           1.000000          -0.116605    0.565181   
CoapplicantIncome        -0.116605           1.000000    0.189218   
LoanAmount                0.565181           0.189218    1.000000   
Loan_Amount_Term         -0.046531          -0.059383    0.036960   
Credit_History           -0.018615           0.011134   -0.000607   

                   Loan_Amount_Term  Credit_History  
ApplicantIncome           -0.046531       -0.018615  
CoapplicantIncome         -0.059383        0.011134  
LoanAmount                 0.036960       -0.000607  
Loan_Amount_Term           1.000000       -0.004705  
Credit_History            -0.004705        1.000000  
